## 模块 4：分省份门店好评率多逻辑排名分析（窗口排名函数）

### 业务需求

1. 基于原始表，按省份维度对门店进行好评率排名，同时展示三种主流排名逻辑的差异与适用场景

2. 仅统计有有效订单的门店，过滤无订单的空门店，确保数据有效性

3. 输出每个省份内门店的好评率排名结果，为区域门店运营评优、标杆学习、优化整改提供数据支撑


### 思路分析

1. 数据清洗与过滤：通过`INNER JOIN`关联销售订单表与门店信息表，仅保留有有效订单的门店，过滤无交易的空门店；使用`DISTINCT`去重，避免同一门店重复出现

2. 逻辑封装：用`WITH CTE`封装门店基础信息子查询，简化主查询逻辑，提升代码可读性与可维护性

3. 分区排名：使用`PARTITION BY 所在省份`实现按省份内部分组，在每个省份独立的排名区间内，按门店好评率降序排序

4. 多逻辑对比：同时使用`ROW_NUMBER()`、`RANK()`、`DENSE_RANK()`三个排名函数，直观展示三种排名逻辑的核心差异

5. 结果排序：最终按省份、好评率降序排序，输出清晰、有序的排名结果

### 核心知识点

- 多表`INNER JOIN`关联，实现有效门店过滤与数据清洗

- `DISTINCT`去重，处理重复门店数据

- `CTE(WITH)`公用表表达式的使用，简化复杂查询逻辑

- 窗口函数`PARTITION BY`分区分组，实现分组内独立排名

- 三大排名函数`ROW_NUMBER()`/`RANK()`/`DENSE_RANK()`的核心区别与适用场景

- 窗口排序逻辑与分区窗口的规范使用

In [1]:
import pandas as pd
import sqlite3

# 1. 连接SQLite数据库
conn = sqlite3.connect("/kaggle/input/datasets/tclaim/retail-sales/retail_sales.db")

# 4. 封装通用SQL执行函数，自动打印表格结果
def query_sql(sql):
    return pd.read_sql(sql, conn)
    
print("数据导入完成，调用 query_sql('SQL语句') 即可执行查询")

数据导入完成，调用 query_sql('SQL语句') 即可执行查询


## SQL 代码

In [2]:
sql = '''
WITH shop_base AS (
    SELECT DISTINCT
        s.门店ID,
        s.门店名称,
        s.所在省份,
        s.`门店好评率(%)`
    FROM 销售订单表 o
    INNER JOIN 门店信息表 s
        ON o.门店ID = s.门店ID
)
SELECT
    所在省份,
    门店名称,
    `门店好评率(%)`,
    ROW_NUMBER() OVER(PARTITION BY 所在省份 ORDER BY `门店好评率(%)` DESC) AS `行号排名`,
    RANK() OVER(PARTITION BY 所在省份 ORDER BY `门店好评率(%)` DESC) AS `跳跃排名`,
    DENSE_RANK() OVER(PARTITION BY 所在省份 ORDER BY `门店好评率(%)` DESC) AS `连续排名`
FROM shop_base
ORDER BY 所在省份, `门店好评率(%)` DESC;
'''
query_sql(sql)

,所在省份,门店名称,门店好评率(%),行号排名,跳跃排名,连续排名
0,云南省,临沧门店352,100,1,1,1
1,云南省,文山壮族苗族自治州门店93,100,2,1,1
2,云南省,西双版纳傣族自治州门店312,85,3,3,2
3,云南省,昆明门店4,80,4,4,3
4,云南省,楚雄彝族自治州门店179,80,5,4,3
...,...,...,...,...,...,...
295,黑龙江省,牡丹江门店324,80,8,8,5
296,黑龙江省,大兴安岭地区门店206,80,9,8,5
297,黑龙江省,鸡西门店124,75,10,10,6
298,黑龙江省,绥化门店59,75,11,10,6


### 结果说明

1. 输出结果：每个省份内所有有有效订单的门店，完整展示省份、门店名称、门店好评率，以及三种排名逻辑的结果

2. 核心差异展示：相同好评率的门店，三种排名函数呈现不同结果

    - `ROW_NUMBER()`：生成连续不重复的唯一序号，适用于需要唯一排名、无并列场景

    - `RANK()`：并列排名后出现跳号，适用于需要区分并列、同时体现排名差距的场景

    - `DENSE_RANK()`：并列排名后不跳号，适用于需要连续排名、无跳号的场景

3. 业务价值：可直接用于区域门店运营评优、高好评门店标杆经验复制、低好评门店优化整改